In [ ]:
# Install dependencies if not already installed
# Run this cell once, then restart the kernel if needed.
 
%pip install transformers torch accelerate sentencepiece

  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached sentencepiece-0.2.1-cp313-cp313-win_amd64.whl.metadata (10 kB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)
Using cached sentencepiece-0.2.1-cp313-cp313-win_amd64.whl (1.1 MB)

   -------------------- ------------------- 1/2 [accelerate]
   ---------------------------------------- 2/2 [accelerate]



In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

c:\Users\Dell\Desktop\Phase3-AI\KDU-2026-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 1: Load tokenizer and model

We will use `distilgpt2`, which is a lightweight pretrained causal language model suitable for text generation experiments.

In [4]:
model_name = "distilgpt2"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(model_name)

# GPT-2 style models do not always have a default pad token.
# For generation stability, set pad_token to eos_token if needed.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("\nModel and tokenizer loaded successfully.")
print("Model name:", model_name)
print("Tokenizer vocab size:", tokenizer.vocab_size)
print("Model class:", model.__class__.__name__)

Loading tokenizer...
Loading model...


Loading weights: 100%|██████████| 76/76 [00:00<00:00, 4787.59it/s]



Model and tokenizer loaded successfully.
Model name: distilgpt2
Tokenizer vocab size: 50257
Model class: GPT2LMHeadModel


## Step 2: Understand tokenization flow

We will demonstrate:

**text → tokens → token IDs**

In [5]:
text = "Artificial intelligence is transforming software development."

print("Original Text:")
print(text)

tokens = tokenizer.tokenize(text)
print("\nTokens:")
print(tokens)

encoded = tokenizer(text, return_tensors="pt")
input_ids = encoded["input_ids"]

print("\nToken IDs:")
print(input_ids)

print("\nToken IDs as Python list:")
print(input_ids[0].tolist())

Original Text:
Artificial intelligence is transforming software development.

Tokens:
['Art', 'ificial', 'Ġintelligence', 'Ġis', 'Ġtransforming', 'Ġsoftware', 'Ġdevelopment', '.']

Token IDs:
tensor([[ 8001,  9542,  4430,   318, 25449,  3788,  2478,    13]])

Token IDs as Python list:
[8001, 9542, 4430, 318, 25449, 3788, 2478, 13]


## Step 3: Decode token IDs back to text

This helps verify that the tokenizer can convert IDs back into readable text.

In [6]:
decoded_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)

print("Decoded Text:")
print(decoded_text)

Decoded Text:
Artificial intelligence is transforming software development.


## Step 4: Basic inference / text generation

We will give the model a prompt and generate text from it.

In [7]:
prompt = "The future of AI in healthcare is"

print("Prompt:")
print(prompt)

inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=True,
        temperature=0.8,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("\nGenerated Output IDs:")
print(output_ids)

print("\nGenerated Text:")
print(generated_text)

Prompt:
The future of AI in healthcare is

Generated Output IDs:
tensor([[  464,  2003,   286,  9552,   287, 11409,   318,  1598,    13,  1081,
           314,  3551,    11,   340,   318,   407,   655,   262,   886,   286,
           262,   995,    11,   475,   772,   262,  3726,    13,  1081,   356,
          1445,  3371,   262,  1306,  7108,   286,  9552,    11,   356,   481,
          1986,   257,  4427,    25,   611,   356,  2314,  6687,   284,  1382,
           257,  1080,   326,   318,   407,   655,   262]])

Generated Text:
The future of AI in healthcare is clear. As I write, it is not just the end of the world, but even the beginning. As we move towards the next phase of AI, we will face a challenge: if we cannot manage to build a system that is not just the


## Step 5: Compare different generation parameter settings

We will test different values of:
- `temperature`
- `top_p`
- `max_new_tokens`

### Notes
- Lower `temperature` usually gives more predictable output
- Higher `temperature` usually gives more random/creative output
- Lower `top_p` makes generation more selective
- Higher `max_new_tokens` gives longer output

In [8]:
def generate_text(prompt, temperature, top_p, max_new_tokens):
    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=True)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return output_ids, generated

In [13]:
experiment_prompt = "Machine learning will change education by"

experiments = [
    {"temperature": 0.7, "top_p": 0.9, "max_new_tokens": 40},
    {"temperature": 1.2, "top_p": 0.9, "max_new_tokens": 40},
    {"temperature": 0.8, "top_p": 0.75, "max_new_tokens": 80},
]

print("Prompt for experiments:")
print(experiment_prompt)

Prompt for experiments:
Machine learning will change education by


## Step 6: Compact comparison view

This cell gives a cleaner side-by-side style output for the generated texts.

In [14]:
results = []

for params in experiments:
    _, generated = generate_text(
        prompt=experiment_prompt,
        temperature=params["temperature"],
        top_p=params["top_p"],
        max_new_tokens=params["max_new_tokens"]
    )
    
    results.append({
        "temperature": params["temperature"],
        "top_p": params["top_p"],
        "max_new_tokens": params["max_new_tokens"],
        "generated_text": generated
    })

for idx, result in enumerate(results, start=1):
    print(f"Experiment {idx}")
    print(f"temperature={result['temperature']}, top_p={result['top_p']}, max_new_tokens={result['max_new_tokens']}")
    print(result["generated_text"])
    print("-" * 100)

Experiment 1
temperature=0.7, top_p=0.9, max_new_tokens=40
Machine learning will change education by improving the way we interact with the world.
----------------------------------------------------------------------------------------------------
Experiment 2
temperature=1.2, top_p=0.9, max_new_tokens=40
Machine learning will change education by encouraging us to use science and design for everyday life. A group of professors in the course developed what is known as an advanced approach to learning and learning, and developed the idea of learning through technology and
----------------------------------------------------------------------------------------------------
Experiment 3
temperature=0.8, top_p=0.75, max_new_tokens=80
Machine learning will change education by creating a new system of learning, learning, and learning systems.
----------------------------------------------------------------------------------------------------


## Step 7: Observations

You should observe that:
- Lower temperature often produces safer and more predictable text
- Higher temperature often produces more diverse but sometimes less coherent text
- Lower top_p restricts token choices more aggressively
- Higher max_new_tokens increases response length

This completes the required deliverables:
- Model loading
- Tokenization
- Inference
- Outputs generated with different parameter settings